In [5]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3801,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")    # [64, 1, 28, 28]
print(f"Labels shape: {labels.shape}")   # [64]
print(f"训练集: {len(train_dataset)}, 测试集: {len(test_dataset)}")  # 60000, 10000

Batch shape: torch.Size([64, 1, 28, 28])
Labels shape: torch.Size([64])
训练集: 60000, 测试集: 10000


In [8]:
class MLP(nn.Module):
    """
    784 -> 256 -> 128 -> 10
    """
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

Using: cpu


In [9]:
model = MLP().to(device)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

参数量: 235,146


In [10]:
# 验证forward pass
dummy = torch.randn(1, 1, 28, 28).to(device)
out = model(dummy)
print(f"输入: {dummy.shape} → 输出: {out.shape}")  # [1,10]

输入: torch.Size([1, 1, 28, 28]) → 输出: torch.Size([1, 10])


In [19]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total



def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(dim = 1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * correct / total
    

In [21]:
NUM_EPOCHS = 10
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | "
      f"{'Test Loss':>9} | {'Test Acc':>8} | {'Time':>5}")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    elapsed = time.time() - start
    print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>9.2f}% | "
          f"{test_loss:>9.4f} | {test_acc:>8.2f}% | {elapsed:>5.1f}s")

Epoch | Train Loss | Train Acc | Test Loss | Test Acc |  Time
-----------------------------------------------------------------
    1 |     0.0829 |     97.47% |    0.0754 |    97.83% |   2.7s
    2 |     0.0702 |     97.72% |    0.0814 |    97.77% |   2.6s
    3 |     0.0659 |     97.88% |    0.0672 |    98.03% |   2.6s
    4 |     0.0570 |     98.14% |    0.0757 |    97.86% |   2.7s
    5 |     0.0520 |     98.33% |    0.0720 |    98.02% |   2.6s
    6 |     0.0479 |     98.52% |    0.0718 |    97.97% |   2.6s
    7 |     0.0473 |     98.44% |    0.0769 |    97.93% |   2.6s
    8 |     0.0404 |     98.66% |    0.0787 |    98.12% |   2.6s
    9 |     0.0430 |     98.62% |    0.0754 |    98.08% |   2.7s
   10 |     0.0388 |     98.75% |    0.0686 |    98.12% |   2.6s


In [22]:
# 看看模型在哪些数字上犯错
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

# 简易混淆矩阵
confusion = torch.zeros(10, 10, dtype=torch.long)
for true, pred in zip(all_labels, all_preds):
    confusion[true][pred] += 1

# 每个数字的准确率
print("每个数字的识别准确率:")
for i in range(10):
    acc = confusion[i][i].item() / confusion[i].sum().item() * 100
    total_wrong = confusion[i].sum().item() - confusion[i][i].item()
    print(f"  数字 {i}: {acc:.1f}%  (错{total_wrong}个)")
# 通常: 1最容易(>99%), 而4/9和3/5容易互相混淆

# 看最常见的混淆对
print("\n最常见的混淆：")
for i in range(10):
    for j in range(10):
        if i != j and confusion[i][j] > 5:
            print(f"  真实{i} → 预测{j}: {confusion[i][j]}次")

每个数字的识别准确率:
  数字 0: 99.1%  (错9个)
  数字 1: 99.6%  (错5个)
  数字 2: 98.3%  (错18个)
  数字 3: 98.9%  (错11个)
  数字 4: 95.9%  (错40个)
  数字 5: 98.0%  (错18个)
  数字 6: 98.2%  (错17个)
  数字 7: 96.5%  (错36个)
  数字 8: 98.4%  (错16个)
  数字 9: 98.2%  (错18个)

最常见的混淆：
  真实4 → 预测9: 28次
  真实5 → 预测3: 8次
  真实7 → 预测1: 7次
  真实7 → 预测2: 14次
  真实7 → 预测9: 8次


In [23]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'test_accuracy': test_acc,
    'architecture': '784-256-128-10 MLP with Dropout(0.2)',
}, 'mnist_mlp_checkpoint.pth')
print(f"✓ 模型已保存, test accuracy: {test_acc:.2f}%")

✓ 模型已保存, test accuracy: 98.12%
